# BKMeeting AI Hub Option 1 NPU Pilots

This notebook adapts the minimal `On_device_Ai.ipynb` example into a repo-specific Qualcomm AI Hub workflow for BKMeeting.

Scope of this notebook:

- stay fully in `python-model-test`
- do not touch Android packaging
- prove compile, profile, and inference on Qualcomm AI Hub first
- split each pilot into:
  - `prepare + compile only`
  - `resolve existing compiled target + run + compare`
- run two pilots:
  - Zipformer encoder-first
  - VPCD model-session-first


## Environment Notes

Before running this notebook, make sure the current environment can already execute the local `python-model-test` bundle helpers.

Minimum practical dependencies:

- `qai-hub`
- `torch`
- `torchaudio`
- `numpy`
- local editable install of this repo if needed

The Zipformer pilot uses the existing repo feature-extraction path, so `torchaudio` must be available.


In [1]:
!pip install qai-hub "qai-hub[torch]"


In [2]:
from pathlib import Path
import subprocess
import sys

sys.path.insert(0, str(Path.cwd() / "src"))

from tools.aihub_option1_pilots import resolve_qai_hub_api_token

API_TOKEN = resolve_qai_hub_api_token(repo_root=Path.cwd())

if not API_TOKEN:
    print("Set QAI_HUB_API_TOKEN in .env or your shell environment before running Qualcomm AI Hub configuration.")
else:
    subprocess.run(["qai-hub", "configure", "--api_token", API_TOKEN], check=True)

!qai-hub list-devices


D:\Anaconda\envs\speech2text\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


+---------------------------------+--------------+----------+---------+---------------------------------------------------+------------------------------------------------------------+
|              Device             |      OS      |  Vendor  |   Type  |                      Chipset                      |                       CLI Invocation                       |
+---------------------------------+--------------+----------+---------+---------------------------------------------------+------------------------------------------------------------+
|     Google Pixel 3 (Family)     |  Android 10  |  Google  |  Phone  |          qualcomm-snapdragon-845, sdm845          |     --device "Google Pixel 3 (Family)" --device-os 10      |
|          Google Pixel 3         |  Android 10  |  Google  |  Phone  |          qualcomm-snapdragon-845, sdm845          |          --device "Google Pixel 3" --device-os 10          |
|         Google Pixel 3a         |  Android 10  |  Google  |  Phone  |    

In [3]:
import sys
from pathlib import Path

import onnxruntime as ort
import qai_hub as hub

from model_bundle.fixtures import read_jsonl
sys.path.insert(0, str(Path.cwd() / "src"))

from tools.aihub_option1_pilots import (
    build_compile_options,
    build_job_options,
    build_option1_runtime_config,
    build_vpcd_input_specs,
    build_vpcd_single_step_calibration_entries,
    build_vpcd_single_step_inputs,
    build_zipformer_encoder_inference_entries,
    build_zipformer_encoder_input_specs,
    compare_output_tensors,
    coerce_inputs_for_compiled_model,
    prepare_vpcd_option1_source_model,
    prepare_zipformer_encoder_option1_source_model,
    resolve_target_model_id,
    resolve_vpcd_fp32_source_model_path,
    resolve_vpcd_pilot_source,
    resolve_zipformer_encoder_pilot_source,
    summarize_vpcd_step_logits,
    write_compile_run_record,
    write_live_run_record,
    write_prepared_artifact_record,
    wrap_single_inference_inputs,
)


In [4]:
DEVICE_NAME = "Samsung Galaxy S24 (Family)"
QAIRT_VERSION = None
RUN_LABEL = "latest"

# Use one stable RUN_LABEL per compiled artifact set, for example: "s24-main" or "debug-2026-05-12".
# If you want to reuse a previous compile without recompiling, keep the same RUN_LABEL so the notebook can
# read build/aihub/records/<pilot>/compile-run-<RUN_LABEL>.json automatically.

# Optional: paste an existing compiled target model id here to bypass compile-record lookup entirely.
# Leave these as None if you want the Resolve step to load the model id from compile-run-<RUN_LABEL>.json.
ZIPFORMER_TARGET_MODEL_ID = None
VPCD_TARGET_MODEL_ID = None

RUNTIME_CONFIG = build_option1_runtime_config(
    device_name=DEVICE_NAME,
    qairt_version=QAIRT_VERSION,
    repo_root=Path.cwd(),
)
job_options = build_job_options(
    compute_unit=RUNTIME_CONFIG.compute_unit,
    qairt_version=RUNTIME_CONFIG.qairt_version,
)

print("device:", RUNTIME_CONFIG.device_name)
print("qairt_version:", RUNTIME_CONFIG.qairt_version)
print("artifact_root:", RUNTIME_CONFIG.artifact_root)
print("record_root:", RUNTIME_CONFIG.record_root)
print("job_options:", job_options)
print("run_label:", RUN_LABEL)
print("zipformer reuse target model id:", ZIPFORMER_TARGET_MODEL_ID)
print("vpcd reuse target model id:", VPCD_TARGET_MODEL_ID)


device: Samsung Galaxy S24 (Family)
qairt_version: None
artifact_root: D:\DS-AI\BKMeeting-Research\python-model-test\build\aihub
record_root: D:\DS-AI\BKMeeting-Research\python-model-test\build\aihub\records
job_options: --compute_unit npu
run_label: latest
zipformer reuse target model id: None
vpcd reuse target model id: None


## How To Use This Notebook

This notebook supports two common workflows.

### Workflow A: Compile From Scratch, Then Run And Compare

Use this when you do **not** already have a compiled target model for the current pilot.

1. Run the setup cells from the top through the config cell.
2. Keep `ZIPFORMER_TARGET_MODEL_ID = None` and `VPCD_TARGET_MODEL_ID = None` unless you want to force a specific target id manually.
3. Choose a `RUN_LABEL` for this compile, for example `latest`, `s24-main`, or `debug-2026-05-12`.
4. For each pilot you want to test, run these sections in order:
   - `Prepare`
   - `Compile Only`
   - `Resolve Existing Compiled Target`
   - `Run And Compare Against The Compiled Target`
   - `Output Inspection`
5. After compile succeeds, the notebook writes `compile-run-<RUN_LABEL>.json` under `build/aihub/records/<pilot>/`.
6. On later days, you can reuse that same compile by keeping the same `RUN_LABEL` and skipping the compile cell.

### Workflow B: Skip Compile, Reuse An Existing Compiled Target, Then Run And Compare

Use this when compile already succeeded earlier and you only want to rerun profile, inference, and output comparison.

You have two ways to reuse an existing compiled target:

1. **Recommended:** keep `RUN_LABEL` the same as the earlier compile and leave `*_TARGET_MODEL_ID = None`.
   - The `Resolve Existing Compiled Target` cell will load the target model id from `compile-run-<RUN_LABEL>.json`.
2. **Manual override:** paste a known target model id into `ZIPFORMER_TARGET_MODEL_ID` or `VPCD_TARGET_MODEL_ID`.
   - This bypasses record lookup and uses that exact compiled target directly.

When reusing a previous compile, run only these sections for the pilot:

- `Prepare`
- `Resolve Existing Compiled Target`
- `Run And Compare Against The Compiled Target`
- `Output Inspection`

You can safely skip the `Compile Only` section in this workflow.


## Pilot 1: Zipformer Encoder-First

This pilot targets the first ASR slice that BKMeeting wants to offload first: the encoder graph.

The current local helper now prepares a dedicated AI Hub upload artifact from the fixed-shape encoder source.

- base source: fixed-shape encoder ONNX
- upload artifact: ORT-optimized + symbolic-shape-prepared + HTP bool-slice rewrite
- local fixtures: current Zipformer bundle sample manifest
- current verified lane: direct `submit_compile_job(...)` on the prepared source model


In [5]:
zipformer_pilot_name = "zipformer_encoder_option1"
zipformer_source = resolve_zipformer_encoder_pilot_source(RUNTIME_CONFIG.repo_root)
zipformer_source_model_path = prepare_zipformer_encoder_option1_source_model(
    zipformer_source,
    output_path=RUNTIME_CONFIG.pilot_artifact_dir(zipformer_pilot_name) / "encoder.aihub.option1.onnx",
)
zipformer_input_specs = build_zipformer_encoder_input_specs(zipformer_source)
zipformer_compile_options = build_compile_options(
    qairt_version=RUNTIME_CONFIG.qairt_version,
    input_specs=zipformer_input_specs,
)
zipformer_raw_inference_inputs = build_zipformer_encoder_inference_entries(zipformer_source)
zipformer_inference_inputs = coerce_inputs_for_compiled_model(
    zipformer_raw_inference_inputs,
    input_specs=zipformer_input_specs,
)
zipformer_prepared_record_path = write_prepared_artifact_record(
    pilot_name=zipformer_pilot_name,
    runtime_config=RUNTIME_CONFIG,
    source_model_path=zipformer_source.source_model_path,
    prepared_model_path=zipformer_source_model_path,
    input_specs=zipformer_input_specs,
    compile_options=zipformer_compile_options,
    run_label=RUN_LABEL,
)

print("zipformer base source model:", zipformer_source.source_model_path)
print("zipformer prepared upload model:", zipformer_source_model_path)
print("zipformer bundle manifest:", zipformer_source.bundle_manifest_path)
print("zipformer input specs:", zipformer_input_specs)
print("zipformer compile options:", zipformer_compile_options)
print("zipformer prepared record:", zipformer_prepared_record_path)
print({name: [value.shape for value in values] for name, values in zipformer_inference_inputs.items()})


Unable to determine if floor(If_597_o0__d0/2) + 501 <= If_597_o0__d0, treat as equal
Cannot determine if floor(If_597_o0__d0/2) - 500 < 0
Unable to determine if floor(If_1168_o0__d0/2) + 251 <= If_1168_o0__d0, treat as equal
Cannot determine if floor(If_1168_o0__d0/2) - 250 < 0
Unable to determine if floor(If_1739_o0__d0/2) + 126 <= If_1739_o0__d0, treat as equal
Cannot determine if floor(If_1739_o0__d0/2) - 125 < 0
Unable to determine if floor(If_2308_o0__d0/2) + 251 <= If_2308_o0__d0, treat as equal
Cannot determine if floor(If_2308_o0__d0/2) - 250 < 0
Unable to determine if floor(If_2877_o0__d0/2) + 501 <= If_2877_o0__d0, treat as equal
Cannot determine if floor(If_2877_o0__d0/2) - 500 < 0


zipformer base source model: D:\DS-AI\BKMeeting-Research\python-model-test\build\quantize\zipformer\qnn_u16u8\fixed_shapes\encoder.fixed.onnx
zipformer prepared upload model: D:\DS-AI\BKMeeting-Research\python-model-test\build\aihub\zipformer_encoder_option1\encoder.aihub.option1.onnx
zipformer bundle manifest: D:\DS-AI\BKMeeting-Research\python-model-test\build\model_bundle\zipformer\qnn_u16u8\bundle_manifest.json
zipformer input specs: {'x': ((1, 2009, 80), 'float32'), 'x_lens': ((1,), 'int64')}
zipformer compile options: --target_runtime precompiled_qnn_onnx --truncate_64bit_io
zipformer prepared record: D:\DS-AI\BKMeeting-Research\python-model-test\build\aihub\records\zipformer_encoder_option1\prepared-artifact-latest.json
{'x': [(1, 2009, 80)], 'x_lens': [(1,)]}


### Zipformer Compile Only

Use this section only when you need to create a **new compiled target model** for Zipformer.

Run this section when:

- this is your first time testing Zipformer on the selected cloud device
- you changed the prepared source model or compile options
- you want a fresh compiled artifact under a new `RUN_LABEL`

After this cell succeeds, save or remember at least one of these:

- `RUN_LABEL`
- `zipformer target model id`
- the record file `build/aihub/records/zipformer_encoder_option1/compile-run-<RUN_LABEL>.json`

If you only want to rerun inference and compare outputs, do **not** rerun this section. Jump to `Resolve Existing Compiled Target` instead.


In [6]:
zipformer_compile_job = hub.submit_compile_job(
    model=zipformer_source_model_path,
    device=hub.Device(RUNTIME_CONFIG.device_name),
    input_specs=zipformer_input_specs,
    options=zipformer_compile_options,
    name="bkmeeting-zipformer-encoder-precompiled-qnn-onnx",
)
zipformer_compiled_target_model = zipformer_compile_job.get_target_model()
zipformer_compile_record_path = write_compile_run_record(
    pilot_name=zipformer_pilot_name,
    runtime_config=RUNTIME_CONFIG,
    compile_options=zipformer_compile_options,
    compile_job=zipformer_compile_job,
    target_model=zipformer_compiled_target_model,
    run_label=RUN_LABEL,
)

print("zipformer compile job:", zipformer_compile_job.url)
print("zipformer target model id:", zipformer_compiled_target_model.model_id)
print("zipformer target model url:", zipformer_compiled_target_model.url)
print("zipformer compile record:", zipformer_compile_record_path)


Uploading encoder.aihub.option1.onnx


100%|██████████| 87.7M/87.7M [00:06<00:00, 13.9MB/s]


Scheduled compile job (j57vdezv5) successfully. To see the status and results:
    https://workbench.aihub.qualcomm.com/jobs/j57vdezv5/

Waiting for compile job (j57vdezv5) completion. Type Ctrl+C to stop waiting at any time.
    ✅ SUCCESS                          
zipformer compile job: https://workbench.aihub.qualcomm.com/jobs/j57vdezv5/
zipformer target model id: mmrlod22m
zipformer target model url: https://workbench.aihub.qualcomm.com/models/mmrlod22m/
zipformer compile record: D:\DS-AI\BKMeeting-Research\python-model-test\build\aihub\records\zipformer_encoder_option1\compile-run-latest.json


### Resolve Existing Compiled Target

This section decides **which compiled Zipformer target model** will be used for profile, inference, and comparison.

It works in two modes:

1. `ZIPFORMER_TARGET_MODEL_ID = None`
   - the notebook reads `build/aihub/records/zipformer_encoder_option1/compile-run-<RUN_LABEL>.json`
   - use this when you want to reuse a previous compile by label
2. `ZIPFORMER_TARGET_MODEL_ID = "..."`
   - the notebook skips record lookup and uses that exact model id directly
   - use this when you copied a target model id from an earlier notebook run or AI Hub page

If this cell fails with a missing record error, it usually means one of these:

- you never ran `Compile Only` for this `RUN_LABEL`
- you changed `RUN_LABEL` and the matching `compile-run-<RUN_LABEL>.json` does not exist yet
- you should paste a known `ZIPFORMER_TARGET_MODEL_ID` manually


In [7]:
zipformer_target_model_id = resolve_target_model_id(
    pilot_name=zipformer_pilot_name,
    runtime_config=RUNTIME_CONFIG,
    explicit_target_model_id=ZIPFORMER_TARGET_MODEL_ID,
    run_label=RUN_LABEL,
)
zipformer_target_model = hub.get_model(zipformer_target_model_id)

print("zipformer resolved target model id:", zipformer_target_model_id)
print("zipformer target model url:", zipformer_target_model.url)


zipformer resolved target model id: mmrlod22m
zipformer target model url: https://workbench.aihub.qualcomm.com/models/mmrlod22m/


### Run And Compare Against The Compiled Target

This is the **fast rerun loop** for Zipformer.

Use this section when:

- compile already exists and you want to rerun on the cloud NPU device
- you want fresh profile/inference jobs without paying compile time again
- you want to compare cloud output against the local CPU baseline again

This section does three things:

1. profile the already-compiled target model on the selected cloud device
2. run inference on the same compiled target model
3. write a fresh `live-run-<RUN_LABEL>.json` record and leave `zipformer_output` ready for the inspection cell

After this cell finishes, run the `Zipformer Output Inspection` cell right below it.


In [8]:
zipformer_profile_job = hub.submit_profile_job(
    model=zipformer_target_model,
    device=hub.Device(RUNTIME_CONFIG.device_name),
    options=job_options,
    name="bkmeeting-zipformer-encoder-profile-npu",
)

zipformer_profile = zipformer_profile_job.download_profile()
zipformer_inference_job = hub.submit_inference_job(
    model=zipformer_target_model,
    device=hub.Device(RUNTIME_CONFIG.device_name),
    inputs=zipformer_inference_inputs,
    options=job_options,
    name="bkmeeting-zipformer-encoder-inference-npu",
)
zipformer_output = zipformer_inference_job.download_output_data()
zipformer_live_record_path = write_live_run_record(
    pilot_name=zipformer_pilot_name,
    runtime_config=RUNTIME_CONFIG,
    compile_options=zipformer_compile_options,
    job_options=job_options,
    compile_job=zipformer_compile_job if "zipformer_compile_job" in globals() else {"status": "reused-target-model"},
    profile_job=zipformer_profile_job,
    inference_job=zipformer_inference_job,
    output_tensors=zipformer_output,
    run_label=RUN_LABEL,
)

print("zipformer profile job:", zipformer_profile_job.url)
print("zipformer inference job:", zipformer_inference_job.url)
print("zipformer live record:", zipformer_live_record_path)
print("zipformer output tensors:", {name: [value.shape for value in values] for name, values in zipformer_output.items()})


Scheduled profile job (jp4jwyo8p) successfully. To see the status and results:
    https://workbench.aihub.qualcomm.com/jobs/jp4jwyo8p/

Waiting for profile job (jp4jwyo8p) completion. Type Ctrl+C to stop waiting at any time.
    ✅ SUCCESS                          


Uploading dataset: 264kB [00:01, 232kB/s]                    <?, ?B/s]


Scheduled inference job (jgkry2nw5) successfully. To see the status and results:
    https://workbench.aihub.qualcomm.com/jobs/jgkry2nw5/

Waiting for inference job (jgkry2nw5) completion. Type Ctrl+C to stop waiting at any time.
    ✅ SUCCESS                          


tmpldt0468n.h5: 100%|██████████| 529k/529k [00:00<00:00, 1.00MB/s]

zipformer profile job: https://workbench.aihub.qualcomm.com/jobs/jp4jwyo8p/
zipformer inference job: https://workbench.aihub.qualcomm.com/jobs/jgkry2nw5/
zipformer live record: D:\DS-AI\BKMeeting-Research\python-model-test\build\aihub\records\zipformer_encoder_option1\live-run-latest.json
zipformer output tensors: {'output_0': [(1, 501, 512)], 'output_1': [(1,)]}


## Zipformer Output Inspection

This pilot only runs the encoder on NPU, so the correctness check here is numerical parity between the cloud NPU encoder output and the local CPU ONNX encoder output for the same input.

The transcript fixture is printed only as context. At this phase, Zipformer still validates encoder tensors, not final RNNT transcripts on device.


In [9]:
zipformer_cpu_inputs = {name: values[0] for name, values in zipformer_raw_inference_inputs.items()}
zipformer_cpu_session = ort.InferenceSession(
    zipformer_source.source_model_path.as_posix(),
    providers=["CPUExecutionProvider"],
)
zipformer_cpu_output_arrays = zipformer_cpu_session.run(None, zipformer_cpu_inputs)
zipformer_cpu_output = {f"output_{index}": [value] for index, value in enumerate(zipformer_cpu_output_arrays)}
zipformer_output_comparison = compare_output_tensors(
    zipformer_cpu_output,
    zipformer_output,
    atol=1e-3,
    rtol=1e-3,
)
zipformer_expected_outputs = read_jsonl(zipformer_source.bundle_manifest_path.parent / "expected_outputs.jsonl")

print("zipformer reference transcript:", zipformer_expected_outputs[0]["text"] if zipformer_expected_outputs else "n/a")
print("zipformer encoder_out_lens (cloud):", zipformer_output["output_1"][0].tolist())
print("zipformer encoder frame preview (cloud):")
print(zipformer_output["output_0"][0][0, :2, :8])
zipformer_output_comparison


zipformer reference transcript: ▁CHÀO▁CÁC▁BẠN▁HÔM▁NAY▁CHÚNG▁TA▁CÙNG▁NHAU▁ĐẾN▁VỚI▁BÀI▁HỌC▁DEP▁LEARNING▁PHẦN▁SỐ▁MƯỜI▁BA▁ĐÁNG▁LÝ▁BÀI▁NÀY▁ĐÃ▁HỌC▁TỪ▁NGÀY▁HAI▁MƯƠI▁MỐT▁THÁNG▁MƯỜI▁HAI▁NĂM▁HAI▁NGHÌN▁KHÔNG▁TRĂM▁HAI▁MƯƠI▁NĂM▁NHƯNG▁VÌ▁NGHỈ▁TẾT▁CHÚNG▁TA▁GIỜ▁LỊCH▁ĐẾN▁NGÀY▁HAI▁MƯƠI▁HAI▁THÁNG▁HAI▁NĂM▁HAI▁NGHÌN▁KHÔNG▁TRĂM▁HAI▁MƯƠI▁SÁU
zipformer encoder_out_lens (cloud): [216]
zipformer encoder frame preview (cloud):
[[-0.47973636  0.69042975  0.09777833  0.189209    0.1859131   0.0300293
   0.5585938   0.05712891]
 [-0.4306641   0.7509766   0.18078615 -0.06140137  0.22790529  0.18579103
   0.5595704  -0.29687503]]


{'output_0': {'reference_dtype': 'float32',
  'candidate_dtype': 'float32',
  'reference_shape': [1, 501, 512],
  'candidate_shape': [1, 501, 512],
  'shape_match': True,
  'allclose': False,
  'max_abs_diff': 0.6959928832948208,
  'mean_abs_diff': 0.009590339702975922},
 'output_1': {'reference_dtype': 'int64',
  'candidate_dtype': 'int32',
  'reference_shape': [1],
  'candidate_shape': [1],
  'shape_match': True,
  'allclose': True,
  'max_abs_diff': 0.0,
  'mean_abs_diff': 0.0}}

## Pilot 2: VPCD Model-Session-First

This pilot targets the punctuation model session while keeping tokenization on the host side.

Important current caveats:

- prefer the repo FP32 export when available, then freeze it to the fixed bundle shapes before upload
- if the source is still QDQ after preparation, compile it directly as the pragmatic fallback lane
- compiled inference inputs must be coerced from `int64` to `int32` when `--truncate_64bit_io` is present


In [10]:
vpcd_pilot_name = "vpcd_option1"
vpcd_source = resolve_vpcd_pilot_source(RUNTIME_CONFIG.repo_root)
vpcd_original_source_model_path = resolve_vpcd_fp32_source_model_path(vpcd_source) or vpcd_source.model_path
vpcd_prepared_source_model_path, vpcd_is_quantized_source = prepare_vpcd_option1_source_model(
    vpcd_source,
    output_path=RUNTIME_CONFIG.pilot_artifact_dir(vpcd_pilot_name) / "model.option1.onnx",
)
vpcd_input_specs = build_vpcd_input_specs(vpcd_source)
vpcd_compile_options = build_compile_options(
    qairt_version=RUNTIME_CONFIG.qairt_version,
    input_specs=vpcd_input_specs,
)
vpcd_calibration_data = build_vpcd_single_step_calibration_entries(vpcd_source, max_samples=4)
vpcd_single_step_inputs = build_vpcd_single_step_inputs(vpcd_source, sample_index=0)
vpcd_raw_inference_inputs = wrap_single_inference_inputs(vpcd_single_step_inputs)
vpcd_inference_inputs = coerce_inputs_for_compiled_model(
    vpcd_raw_inference_inputs,
    input_specs=vpcd_input_specs,
)
vpcd_prepared_record_path = write_prepared_artifact_record(
    pilot_name=vpcd_pilot_name,
    runtime_config=RUNTIME_CONFIG,
    source_model_path=vpcd_original_source_model_path,
    prepared_model_path=vpcd_prepared_source_model_path,
    input_specs=vpcd_input_specs,
    compile_options=vpcd_compile_options,
    run_label=RUN_LABEL,
)

print("vpcd source model:", vpcd_original_source_model_path)
print("vpcd prepared upload model:", vpcd_prepared_source_model_path)
print("vpcd input specs:", vpcd_input_specs)
print("vpcd compile options:", vpcd_compile_options)
print("vpcd quantized source:", vpcd_is_quantized_source)
print("vpcd prepared record:", vpcd_prepared_record_path)
print({name: [value.shape for value in values] for name, values in vpcd_inference_inputs.items()})


vpcd source model: D:\DS-AI\BKMeeting-Research\python-model-test\assets\vietnamese-punc-cap-denorm-v1\onnx\model.fp32.onnx
vpcd prepared upload model: D:\DS-AI\BKMeeting-Research\python-model-test\build\aihub\vpcd_option1\model.option1.onnx
vpcd input specs: {'input_ids': ((1, 1024), 'int64'), 'attention_mask': ((1, 1024), 'int64'), 'decoder_input_ids': ((1, 128), 'int64'), 'decoder_attention_mask': ((1, 128), 'int64')}
vpcd compile options: --target_runtime precompiled_qnn_onnx --truncate_64bit_io
vpcd quantized source: False
vpcd prepared record: D:\DS-AI\BKMeeting-Research\python-model-test\build\aihub\records\vpcd_option1\prepared-artifact-latest.json
{'input_ids': [(1, 1024)], 'attention_mask': [(1, 1024)], 'decoder_input_ids': [(1, 128)], 'decoder_attention_mask': [(1, 128)]}


### VPCD Compile Only

Use this section only when you need to create a **new compiled target model** for VPCD.

Run this section when:

- this is your first time testing VPCD on the selected cloud device
- you changed the prepared source model, quantize step, or compile options
- you want a fresh compiled artifact under a new `RUN_LABEL`

After this cell succeeds, save or remember at least one of these:

- `RUN_LABEL`
- `vpcd target model id`
- the record file `build/aihub/records/vpcd_option1/compile-run-<RUN_LABEL>.json`

If you only want to rerun inference and compare outputs, do **not** rerun this section. Jump to `Resolve Existing Compiled Target` instead.


In [11]:
vpcd_quantize_job = None
if vpcd_is_quantized_source:
    vpcd_compile_input_model = vpcd_prepared_source_model_path
    print("VPCD source is already QDQ. Compiling directly for the current AI Hub pilot.")
else:
    vpcd_quantize_job = hub.submit_quantize_job(
        model=vpcd_prepared_source_model_path,
        calibration_data=vpcd_calibration_data,
        name="bkmeeting-vpcd-quantize",
    )
    vpcd_compile_input_model = vpcd_quantize_job.get_target_model()
    print("vpcd quantize job:", vpcd_quantize_job.url)

vpcd_compile_job = hub.submit_compile_job(
    model=vpcd_compile_input_model,
    device=hub.Device(RUNTIME_CONFIG.device_name),
    input_specs=vpcd_input_specs,
    options=vpcd_compile_options,
    name="bkmeeting-vpcd-precompiled-qnn-onnx",
)
vpcd_compiled_target_model = vpcd_compile_job.get_target_model()
vpcd_compile_record_path = write_compile_run_record(
    pilot_name=vpcd_pilot_name,
    runtime_config=RUNTIME_CONFIG,
    compile_options=vpcd_compile_options,
    compile_job=vpcd_compile_job,
    target_model=vpcd_compiled_target_model,
    run_label=RUN_LABEL,
)

print("vpcd compile job:", vpcd_compile_job.url)
print("vpcd target model id:", vpcd_compiled_target_model.model_id)
print("vpcd target model url:", vpcd_compiled_target_model.url)
print("vpcd compile record:", vpcd_compile_record_path)


Uploading model.option1.onnx part 1 of 2


100%|██████████| 1.00G/1.00G [01:00<00:00, 17.8MB/s]


Uploading model.option1.onnx part 2 of 2


100%|██████████| 643M/643M [00:36<00:00, 18.3MB/s] 
Uploading dataset: 36.6kB [00:00, 47.6kB/s]                   <?, ?B/s]


Scheduled quantize job (jglekyrjp) successfully. To see the status and results:
    https://workbench.aihub.qualcomm.com/jobs/jglekyrjp/

Waiting for quantize job (jglekyrjp) completion. Type Ctrl+C to stop waiting at any time.
    ✅ SUCCESS                          
vpcd quantize job: https://workbench.aihub.qualcomm.com/jobs/jglekyrjp/
Scheduled compile job (jp1q8oz7g) successfully. To see the status and results:
    https://workbench.aihub.qualcomm.com/jobs/jp1q8oz7g/

Waiting for compile job (jp1q8oz7g) completion. Type Ctrl+C to stop waiting at any time.
    ✅ SUCCESS                          
vpcd compile job: https://workbench.aihub.qualcomm.com/jobs/jp1q8oz7g/
vpcd target model id: mm5y26v9n
vpcd target model url: https://workbench.aihub.qualcomm.com/models/mm5y26v9n/
vpcd compile record: D:\DS-AI\BKMeeting-Research\python-model-test\build\aihub\records\vpcd_option1\compile-run-latest.json


### Resolve Existing Compiled Target

This section decides **which compiled VPCD target model** will be used for profile, inference, and comparison.

It works in two modes:

1. `VPCD_TARGET_MODEL_ID = None`
   - the notebook reads `build/aihub/records/vpcd_option1/compile-run-<RUN_LABEL>.json`
   - use this when you want to reuse a previous compile by label
2. `VPCD_TARGET_MODEL_ID = "..."`
   - the notebook skips record lookup and uses that exact model id directly
   - use this when you copied a target model id from an earlier notebook run or AI Hub page

If this cell fails with a missing record error, it usually means one of these:

- you never ran `Compile Only` for this `RUN_LABEL`
- you changed `RUN_LABEL` and the matching `compile-run-<RUN_LABEL>.json` does not exist yet
- you should paste a known `VPCD_TARGET_MODEL_ID` manually


In [12]:
vpcd_target_model_id = resolve_target_model_id(
    pilot_name=vpcd_pilot_name,
    runtime_config=RUNTIME_CONFIG,
    explicit_target_model_id=VPCD_TARGET_MODEL_ID,
    run_label=RUN_LABEL,
)
vpcd_target_model = hub.get_model(vpcd_target_model_id)

print("vpcd resolved target model id:", vpcd_target_model_id)
print("vpcd target model url:", vpcd_target_model.url)


vpcd resolved target model id: mm5y26v9n
vpcd target model url: https://workbench.aihub.qualcomm.com/models/mm5y26v9n/


### Run And Compare Against The Compiled Target

This is the **fast rerun loop** for VPCD.

Use this section when:

- compile already exists and you want to rerun on the cloud NPU device
- you want fresh profile/inference jobs without paying compile time again
- you want to compare cloud output against the local CPU baseline again

This section does three things:

1. profile the already-compiled target model on the selected cloud device
2. run inference on the same compiled target model
3. write a fresh `live-run-<RUN_LABEL>.json` record and leave `vpcd_output` ready for the inspection cell

After this cell finishes, run the `VPCD Output Inspection` cell right below it.


In [13]:
vpcd_profile_job = hub.submit_profile_job(
    model=vpcd_target_model,
    device=hub.Device(RUNTIME_CONFIG.device_name),
    options=job_options,
    name="bkmeeting-vpcd-profile-npu",
)

vpcd_profile = vpcd_profile_job.download_profile()
vpcd_inference_job = hub.submit_inference_job(
    model=vpcd_target_model,
    device=hub.Device(RUNTIME_CONFIG.device_name),
    inputs=vpcd_inference_inputs,
    options=job_options,
    name="bkmeeting-vpcd-inference-npu",
)
vpcd_output = vpcd_inference_job.download_output_data()
vpcd_live_record_path = write_live_run_record(
    pilot_name=vpcd_pilot_name,
    runtime_config=RUNTIME_CONFIG,
    compile_options=vpcd_compile_options,
    job_options=job_options,
    compile_job=vpcd_compile_job if "vpcd_compile_job" in globals() else {"status": "reused-target-model"},
    profile_job=vpcd_profile_job,
    inference_job=vpcd_inference_job,
    output_tensors=vpcd_output,
    run_label=RUN_LABEL,
)

print("vpcd profile job:", vpcd_profile_job.url)
print("vpcd inference job:", vpcd_inference_job.url)
print("vpcd live record:", vpcd_live_record_path)
print("vpcd output tensors:", {name: [value.shape for value in values] for name, values in vpcd_output.items()})


Scheduled profile job (jpyvd83lp) successfully. To see the status and results:
    https://workbench.aihub.qualcomm.com/jobs/jpyvd83lp/

Waiting for profile job (jpyvd83lp) completion. Type Ctrl+C to stop waiting at any time.
    ✅ SUCCESS                          


Uploading dataset: 25.3kB [00:00, 37.7kB/s]                   <?, ?B/s]


Scheduled inference job (jp8w7jyop) successfully. To see the status and results:
    https://workbench.aihub.qualcomm.com/jobs/jp8w7jyop/

Waiting for inference job (jp8w7jyop) completion. Type Ctrl+C to stop waiting at any time.
    ✅ SUCCESS                          


tmpp6arvpsd.h5: 100%|██████████| 1.12M/1.12M [00:00<00:00, 1.57MB/s]


vpcd profile job: https://workbench.aihub.qualcomm.com/jobs/jpyvd83lp/
vpcd inference job: https://workbench.aihub.qualcomm.com/jobs/jp8w7jyop/
vpcd live record: D:\DS-AI\BKMeeting-Research\python-model-test\build\aihub\records\vpcd_option1\live-run-latest.json
vpcd output tensors: {'output_0': [(1, 128, 40030)], 'output_1': [(1, 1024, 1024)]}


## VPCD Output Inspection

This pilot checks both numerical parity and decoder-step behavior. The cell below compares cloud NPU logits against a local CPU ONNX baseline and then shows the top token candidates at the active decoder position.


In [14]:
vpcd_cpu_inputs = {name: value for name, value in vpcd_single_step_inputs.items()}
vpcd_cpu_session = ort.InferenceSession(
    vpcd_prepared_source_model_path.as_posix(),
    providers=["CPUExecutionProvider"],
)
vpcd_cpu_output_arrays = vpcd_cpu_session.run(None, vpcd_cpu_inputs)
vpcd_cpu_output = {f"output_{index}": [value] for index, value in enumerate(vpcd_cpu_output_arrays)}
vpcd_output_comparison = compare_output_tensors(
    vpcd_cpu_output,
    vpcd_output,
    atol=1e-2,
    rtol=1e-2,
)
vpcd_golden_sample = read_jsonl(vpcd_source.golden_samples_path)[0]
vpcd_next_token_summary = summarize_vpcd_step_logits(
    vpcd_output["output_0"][0],
    vpcd_single_step_inputs["decoder_attention_mask"],
    top_k=5,
)

print("vpcd raw_text:", vpcd_golden_sample["raw_text"])
print("vpcd expected_output:", vpcd_golden_sample["expected_output"])
print("vpcd active decoder index:", vpcd_next_token_summary["active_index"])
print("vpcd top next-token candidates:")
for item in vpcd_next_token_summary["top_tokens"]:
    print(item)
vpcd_output_comparison


vpcd raw_text: hôm nay là buổi nhậm chức của tôi phước thành
vpcd expected_output: Hôm nay là buổi nhậm chức của tôi - Phước Thành.
vpcd active decoder index: 0
vpcd top next-token candidates:
{'token_id': 0, 'score': 49.22596740722656}
{'token_id': 6609, 'score': 49.22596740722656}
{'token_id': 17786, 'score': 49.22596740722656}
{'token_id': 947, 'score': 49.22596740722656}
{'token_id': 889, 'score': 49.22596740722656}


{'output_0': {'reference_dtype': 'float32',
  'candidate_dtype': 'float32',
  'reference_shape': [1, 128, 40030],
  'candidate_shape': [1, 128, 40030],
  'shape_match': True,
  'allclose': False,
  'max_abs_diff': 61.031280517578125,
  'mean_abs_diff': 45.44200608524837},
 'output_1': {'reference_dtype': 'float32',
  'candidate_dtype': 'float32',
  'reference_shape': [1, 1024, 1024],
  'candidate_shape': [1, 1024, 1024],
  'shape_match': True,
  'allclose': False,
  'max_abs_diff': 4.1272929310798645,
  'mean_abs_diff': 0.3763680103623363}}

## After The Notebook Runs

This notebook now leaves behind a deterministic Phase 2 evidence trail for each pilot:

- prepared upload artifact under `build/aihub/<pilot>/`
- prepared artifact record under `build/aihub/records/<pilot>/prepared-artifact-<RUN_LABEL>.json`
- compile-only record under `build/aihub/records/<pilot>/compile-run-<RUN_LABEL>.json`
- live run record under `build/aihub/records/<pilot>/live-run-<RUN_LABEL>.json`
- AI Hub job URLs printed in the execution cells

Recommended habit:

- use one stable `RUN_LABEL` for one compiled artifact set
- when you want to reuse compile later, keep the same `RUN_LABEL` and skip the `Compile Only` cell
- if you already know a target model id, paste it into `*_TARGET_MODEL_ID` and go straight to `Resolve Existing Compiled Target`

Phase 2 stops at reproducibility and handoff quality. The next phase starts only after these records are present and reviewed.


In [15]:
print("runtime record root:", RUNTIME_CONFIG.record_root)
print("zipformer prepared record:", zipformer_prepared_record_path)
print("zipformer compile record:", RUNTIME_CONFIG.pilot_record_dir(zipformer_pilot_name) / f"compile-run-{RUN_LABEL}.json")
print("zipformer live record:", RUNTIME_CONFIG.pilot_record_dir(zipformer_pilot_name) / f"live-run-{RUN_LABEL}.json")
print("vpcd prepared record:", vpcd_prepared_record_path)
print("vpcd compile record:", RUNTIME_CONFIG.pilot_record_dir(vpcd_pilot_name) / f"compile-run-{RUN_LABEL}.json")
print("vpcd live record:", RUNTIME_CONFIG.pilot_record_dir(vpcd_pilot_name) / f"live-run-{RUN_LABEL}.json")


runtime record root: D:\DS-AI\BKMeeting-Research\python-model-test\build\aihub\records
zipformer prepared record: D:\DS-AI\BKMeeting-Research\python-model-test\build\aihub\records\zipformer_encoder_option1\prepared-artifact-latest.json
zipformer compile record: D:\DS-AI\BKMeeting-Research\python-model-test\build\aihub\records\zipformer_encoder_option1\compile-run-latest.json
zipformer live record: D:\DS-AI\BKMeeting-Research\python-model-test\build\aihub\records\zipformer_encoder_option1\live-run-latest.json
vpcd prepared record: D:\DS-AI\BKMeeting-Research\python-model-test\build\aihub\records\vpcd_option1\prepared-artifact-latest.json
vpcd compile record: D:\DS-AI\BKMeeting-Research\python-model-test\build\aihub\records\vpcd_option1\compile-run-latest.json
vpcd live record: D:\DS-AI\BKMeeting-Research\python-model-test\build\aihub\records\vpcd_option1\live-run-latest.json
